# ML-06 / ML-07 Feature Combination Comparison

## 1. 코드 전체 요약

이 노트북은 `ml-06` fixed-param 피처 축소 실험과 `ml-07` validation-only Optuna tuned 실험을 같은 화면에서 비교하기 위한 로컬 시각화 자료다. 모델을 다시 학습하거나 test set을 읽지 않고, 이미 저장된 validation artifact만 읽어서 피처 조합별 metric table, grouped bar chart, feature count 대비 성능, TP/FP trade-off, learning curve, confusion matrix, feature importance, feature association, score profile을 표시한다.

## 2. 데이터 흐름 요약

- 비교 대상 run은 이 노트북의 `CANDIDATES` manifest에 고정되어 있다.
- 각 run은 `ml/<ml-folder>/ml_outputs/<run-id>/<experiment-id>__<run-id>__<model-run-id>_*` 형식의 산출물로 연결된다.
- `metrics_val.json`, optional `metrics_train.json`, `train_summary.json`, `threshold.json`, `confusion_matrix_val.csv`, `feature_importance.csv`, `feature_assoc_mixed_val.json` 또는 `feature_assoc_mixed_train.json`을 읽는다.
- `prediction_scores_val.parquet`는 threshold sweep을 켰을 때만 후보별로 한 파일씩 순차 로드한다.
- 기존 `ml/00_plot_prediction_score_learning_curves.ipynb`는 수정하지 않는다.

## 3. 확인 필요 사항

- 이 노트북은 validation-only 비교용이다. ML-07/ML-08 정책에 맞게 test artifact를 읽거나 test 기준 threshold를 고르지 않는다.
- `docs/프로젝트계획서_0429.pdf`가 없으면 Part 2 보완안과 ML-06/ML-07 결과 문서를 기준으로 해석한다.


In [1]:
# ============================================================
# 사용자 설정 영역
# ============================================================
from __future__ import annotations

INCLUDE_ML06_EXCLUDED = False
ENABLE_THRESHOLD_SWEEP = True
RECALL_FLOORS = [0.40, 0.42, 0.45]

TOP_N_FEATURES = 20
FEATURE_IMPORTANCE_SORT = "high"
SHOW_FEATURE_CORRELATION = True
SHOW_FILE_MANIFEST = True
SHOW_SCORE_PROFILE = True
SHOW_RUN_DETAILS = True
SHOW_HYPERPARAMETERS = False

LEARNING_CURVE_METRICS = ["LogLoss", "AUPRC"]
RUN_DETAIL_LIMIT = None
SKIP_INVALID_RUNS = True


In [2]:
# ============================================================
# 공통 import와 경로 탐색
# ============================================================
import json
import os
import re
from pathlib import Path
from typing import Any, Optional

MPLCONFIGDIR = Path("/tmp/matplotlib-codex-cache")
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import pandas as pd
from IPython.display import Markdown, display

try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    px = None
    go = None
    HAS_PLOTLY = False
    import matplotlib.pyplot as plt
    print("plotly가 없어 matplotlib fallback으로 표시합니다.")


def find_repo_root(start: Path) -> Path:
    resolved = start.resolve()
    for candidate in (resolved, *resolved.parents):
        if (candidate / "ml").is_dir() and (candidate / "docs").is_dir():
            return candidate
    raise FileNotFoundError(f"Cannot find repository root from {start}")


BASE_DIR = find_repo_root(Path.cwd())
ML_DIR = BASE_DIR / "ml"
DOCS_DIR = BASE_DIR / "docs"
PLAN_PDF_PATH = DOCS_DIR / "프로젝트계획서_0429.pdf"
PART2_PLAN_PATH = DOCS_DIR / "프로젝트계획서_260502_part2_보완안.md"
ML06_REPORT_PATH = DOCS_DIR / "ml06_practical_feature_reduction_result_report_2026-06-02.md"
ML07_REPORT_PATH = DOCS_DIR / "ml07_optuna_validation_consolidated_report_2026-06-04.md"

print(f"BASE_DIR = {BASE_DIR}")
print(f"PLAN_PDF exists = {PLAN_PDF_PATH.exists()} ({PLAN_PDF_PATH})")
print(f"PART2_PLAN exists = {PART2_PLAN_PATH.exists()} ({PART2_PLAN_PATH})")


BASE_DIR = /mnt/d/AML_git/Graph_AML
PLAN_PDF exists = False (/mnt/d/AML_git/Graph_AML/docs/프로젝트계획서_0429.pdf)
PART2_PLAN exists = True (/mnt/d/AML_git/Graph_AML/docs/프로젝트계획서_260502_part2_보완안.md)


In [3]:
# ============================================================
# 비교 대상 manifest
# artifact prefix 규칙: <experiment_id>__<run_id>__<model_run_id>_<suffix>
# ============================================================
BASE_CANDIDATES = [
    {"phase": "ML-06 fixed-param", "phase_order": 10, "phase_kind": "fixed", "candidate": "full d02", "family": "full", "candidate_order": 10, "ml_folder": "ml-06", "experiment_id": "ml_06", "run_id": "r00", "model_run_id": "d02-maxdepth4-fixparam-report", "description": "ML-06 full fixed-param reference", "include_by_default": True},
    {"phase": "ML-06 fixed-param", "phase_order": 10, "phase_kind": "fixed", "candidate": "practical", "family": "practical", "candidate_order": 20, "ml_folder": "ml-06", "experiment_id": "ml_06", "run_id": "r00", "model_run_id": "d02-maxdepth4-fixparam-practical", "description": "ML-06 practical reduced fixed-param baseline", "include_by_default": True},
    {"phase": "ML-06 fixed-param", "phase_order": 10, "phase_kind": "fixed", "candidate": "top10", "family": "top10", "candidate_order": 30, "ml_folder": "ml-06", "experiment_id": "ml_06", "run_id": "r00", "model_run_id": "d02-maxdepth4-fixparam-practical-plus-rev27-gain-top10", "description": "ML-06 practical + rev27 gain top10 fixed-param", "include_by_default": True},
    {"phase": "ML-06 fixed-param", "phase_order": 10, "phase_kind": "fixed", "candidate": "top12", "family": "top12", "candidate_order": 40, "ml_folder": "ml-06", "experiment_id": "ml_06", "run_id": "r00", "model_run_id": "d02-maxdepth4-fixparam-practical-plus-rev27-gain-top12", "description": "ML-06 practical + rev27 gain top12 fixed-param", "include_by_default": True},
    {"phase": "ML-06 fixed-param", "phase_order": 10, "phase_kind": "fixed", "candidate": "fanin_only", "family": "top12-plus-fanin", "candidate_order": 50, "ml_folder": "ml-06", "experiment_id": "ml_06", "run_id": "r00", "model_run_id": "d02-maxdepth4-fixparam-practical_top12_plus_fanin_only", "description": "ML-06 top12 + remaining fan-in features fixed-param", "include_by_default": True},
    {"phase": "ML-06 fixed-param", "phase_order": 10, "phase_kind": "fixed", "candidate": "acc_only", "family": "top12-plus-acc", "candidate_order": 60, "ml_folder": "ml-06", "experiment_id": "ml_06", "run_id": "r00", "model_run_id": "d02-maxdepth4-fixparam-practical_top12_plus_acc_only", "description": "ML-06 top12 + remaining accountstats features fixed-param", "include_by_default": True},
    {"phase": "ML-07 tuned", "phase_order": 20, "phase_kind": "tuned", "candidate": "top10", "family": "top10", "candidate_order": 130, "ml_folder": "ml-07", "experiment_id": "ml_07", "run_id": "r00", "model_run_id": "ml07-d00-md4-top10-optuna-f1-t25", "description": "ML-07 top10 Optuna validation-only tuned", "include_by_default": True},
    {"phase": "ML-07 tuned", "phase_order": 20, "phase_kind": "tuned", "candidate": "top12", "family": "top12", "candidate_order": 140, "ml_folder": "ml-07", "experiment_id": "ml_07", "run_id": "r00", "model_run_id": "ml07-d00-md4-top12-optuna-f1-t25", "description": "ML-07 top12 Optuna validation-only tuned", "include_by_default": True},
    {"phase": "ML-07 tuned", "phase_order": 20, "phase_kind": "tuned", "candidate": "top12-plus-fanin", "family": "top12-plus-fanin", "candidate_order": 150, "ml_folder": "ml-07", "experiment_id": "ml_07", "run_id": "r00", "model_run_id": "ml07-d00-md4-top12-plus-fanin-optuna-f1-t25", "description": "ML-07 top12 plus fan-in Optuna validation-only tuned", "include_by_default": True},
    {"phase": "ML-07 tuned", "phase_order": 20, "phase_kind": "tuned", "candidate": "top12-plus-acc", "family": "top12-plus-acc", "candidate_order": 160, "ml_folder": "ml-07", "experiment_id": "ml_07", "run_id": "r00", "model_run_id": "ml07-d00-md4-top12-plus-acc-optuna-f1-t25", "description": "ML-07 top12 plus accountstats Optuna validation-only tuned", "include_by_default": True},
]

ML06_EXCLUDED_CANDIDATES = [
    {"phase": "ML-06 excluded", "phase_order": 15, "phase_kind": "excluded", "candidate": "top6", "family": "top6", "candidate_order": 70, "ml_folder": "ml-06", "experiment_id": "ml_06", "run_id": "r00", "model_run_id": "d02-maxdepth4-fixparam-practical-plus-rev27-gain-top6.csv", "description": "ML-06 excluded: top6 collapsed ranking", "include_by_default": False},
    {"phase": "ML-06 excluded", "phase_order": 15, "phase_kind": "excluded", "candidate": "top8", "family": "top8", "candidate_order": 80, "ml_folder": "ml-06", "experiment_id": "ml_06", "run_id": "r00", "model_run_id": "d02-maxdepth4-fixparam-practical-plus-rev27-gain-top8", "description": "ML-06 excluded: top8 collapsed ranking", "include_by_default": False},
    {"phase": "ML-06 excluded", "phase_order": 15, "phase_kind": "excluded", "candidate": "next1", "family": "next1", "candidate_order": 90, "ml_folder": "ml-06", "experiment_id": "ml_06", "run_id": "r00", "model_run_id": "d02-maxdepth4-fixparam-practical_top12_plus_next1", "description": "ML-06 excluded: top12 + next1 underperformed", "include_by_default": False},
    {"phase": "ML-06 excluded", "phase_order": 15, "phase_kind": "excluded", "candidate": "next3", "family": "next3", "candidate_order": 100, "ml_folder": "ml-06", "experiment_id": "ml_06", "run_id": "r00", "model_run_id": "d02-maxdepth4-fixparam-practical_top12_plus_next3", "description": "ML-06 excluded: top12 + next3 collapsed ranking", "include_by_default": False},
    {"phase": "ML-06 excluded", "phase_order": 15, "phase_kind": "excluded", "candidate": "rev2", "family": "rev2", "candidate_order": 110, "ml_folder": "ml-06", "experiment_id": "ml_06", "run_id": "r00", "model_run_id": "d02-maxdepth4-fixparam-practical-rev2", "description": "ML-06 excluded: rev2 early-stopped and degraded AP/F1", "include_by_default": False},
]

CANDIDATES = BASE_CANDIDATES + (ML06_EXCLUDED_CANDIDATES if INCLUDE_ML06_EXCLUDED else [])

ARTIFACT_SUFFIXES = {
    "metrics": "metrics_val.json",
    "metrics_train": "metrics_train.json",
    "train_summary": "train_summary.json",
    "confusion_matrix": "confusion_matrix_val.csv",
    "feature_importance": "feature_importance.csv",
    "feature_assoc_val": "feature_assoc_mixed_val.json",
    "feature_assoc_train": "feature_assoc_mixed_train.json",
    "threshold": "threshold.json",
    "feature_columns": "feature_columns.json",
    "prediction_scores_val": "prediction_scores_val.parquet",
}


In [4]:
# ============================================================
# Artifact loader와 metric helper
# ============================================================
def read_json(path: Path) -> Any:
    try:
        with path.open("r", encoding="utf-8") as file:
            return json.load(file)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON: {path}") from exc


def artifact_prefix(rep: dict[str, Any]) -> str:
    return f"{rep['experiment_id']}__{rep['run_id']}__{rep['model_run_id']}"


def run_output_dir(rep: dict[str, Any]) -> Path:
    return ML_DIR / rep["ml_folder"] / "ml_outputs" / rep["run_id"]


def artifact_path(output_dir: Path, prefix: str, suffix: str) -> Path:
    return output_dir / f"{prefix}_{suffix}"


def display_label(rep: dict[str, Any]) -> str:
    return f"{rep['phase']} / {rep['candidate']}"


def metrics_payload(metrics_raw: Any) -> dict[str, Any]:
    if not isinstance(metrics_raw, dict):
        return {}
    metrics = metrics_raw.get("metrics", metrics_raw)
    return metrics if isinstance(metrics, dict) else {}


def train_summary_payload(ml: dict[str, Any]) -> dict[str, Any]:
    payload = ml.get("train_summary", {})
    return payload if isinstance(payload, dict) else {}


def threshold_payload(ml: dict[str, Any]) -> dict[str, Any]:
    payload = ml.get("threshold", {})
    return payload if isinstance(payload, dict) else {}


def metric_value(metrics: dict[str, Any], train_summary: dict[str, Any], metric: str) -> Any:
    if metric == "F1":
        return metrics.get("f1")
    if metric == "AUPRC":
        return metrics.get("average_precision") or train_summary.get("best_score")
    if metric == "Recall":
        return metrics.get("recall")
    if metric == "Precision":
        return metrics.get("precision")
    if metric == "Threshold":
        return metrics.get("threshold")
    return None


def confusion_counts(ml: dict[str, Any]) -> Optional[dict[str, int]]:
    conf_mat = ml.get("confusion_matrix")
    if isinstance(conf_mat, pd.DataFrame) and not conf_mat.empty:
        row = conf_mat.iloc[0]
        return {name: int(row.get(name, 0)) for name in ("tn", "fp", "fn", "tp")}
    metrics_raw = ml.get("metrics", {})
    metrics = metrics_payload(metrics_raw)
    if all(name in metrics for name in ("tn", "fp", "fn", "tp")):
        return {name: int(metrics.get(name, 0)) for name in ("tn", "fp", "fn", "tp")}
    conf = metrics_raw.get("confusion_matrix", {}) if isinstance(metrics_raw, dict) else {}
    if all(name in conf for name in ("tn", "fp", "fn", "tp")):
        return {name: int(conf.get(name, 0)) for name in ("tn", "fp", "fn", "tp")}
    return None


def eval_keys_for_train_val(evals: dict[str, Any]) -> tuple[Optional[str], Optional[str]]:
    keys = list(evals.keys())
    if "train" in evals and "val" in evals:
        return "train", "val"
    if "validation_0" in evals and "validation_1" in evals:
        return "validation_0", "validation_1"
    if len(keys) >= 2:
        return keys[0], keys[1]
    return None, None


def available_learning_curves(train_summary: dict[str, Any]) -> tuple[dict[str, tuple[tuple[float, ...], tuple[float, ...]]], str]:
    diag = train_summary.get("xgboost_diagnostics") or {}
    evals = diag.get("evals_result") or {}
    source = "xgboost_diagnostics.evals_result"
    if not isinstance(evals, dict) or not evals:
        learning_curve = train_summary.get("learning_curve") or {}
        curves = learning_curve.get("curves") or {}
        if isinstance(curves, dict) and curves.get("train") and curves.get("val"):
            evals = {"train": curves["train"], "val": curves["val"]}
            source = "learning_curve.curves"
        else:
            return {}, source
    train_key, val_key = eval_keys_for_train_val(evals)
    if train_key is None or val_key is None:
        return {}, source
    train_metrics = evals.get(train_key) or {}
    val_metrics = evals.get(val_key) or {}
    available = {}
    for metric, train_vals in train_metrics.items():
        if metric in val_metrics:
            available[str(metric)] = (tuple(train_vals), tuple(val_metrics[metric]))
    return available, source


def dashboard_metric_key(metric_label: str) -> str:
    return {"LogLoss": "logloss", "AUPRC": "aucpr"}.get(metric_label, metric_label)


def optuna_summary_path(rep: dict[str, Any], output_dir: Path) -> Path:
    return output_dir / f"optuna__{rep['model_run_id']}" / "full" / "optuna_study_summary.json"


def load_optional_artifacts(rep: dict[str, Any]) -> dict[str, Any]:
    prefix = artifact_prefix(rep)
    output_dir = run_output_dir(rep)
    result = {"rep": rep, "prefix": prefix, "output_dir": output_dir, "label": display_label(rep), "ml": {}, "files": [], "missing": []}
    for key, suffix in ARTIFACT_SUFFIXES.items():
        path = artifact_path(output_dir, prefix, suffix)
        if not path.exists():
            result["missing"].append({"artifact": key, "path": str(path)})
            if key == "metrics_train":
                result["ml"][key] = None
            continue
        result["files"].append({"artifact": key, "path": str(path), "size_mb": path.stat().st_size / 1024 / 1024})
        if key == "prediction_scores_val":
            result["ml"]["prediction_scores_path"] = path
        elif key in {"metrics", "metrics_train", "train_summary", "threshold", "feature_columns", "feature_assoc_val", "feature_assoc_train"}:
            result["ml"][key] = read_json(path)
        elif key in {"confusion_matrix", "feature_importance"}:
            result["ml"][key] = pd.read_csv(path)
    optuna_path = optuna_summary_path(rep, output_dir)
    if optuna_path.exists():
        result["files"].append({"artifact": "optuna_summary", "path": str(optuna_path), "size_mb": optuna_path.stat().st_size / 1024 / 1024})
        result["ml"]["optuna_summary"] = read_json(optuna_path)
    if "feature_assoc_val" in result["ml"]:
        result["ml"]["feature_assoc"] = result["ml"]["feature_assoc_val"]
    elif "feature_assoc_train" in result["ml"]:
        result["ml"]["feature_assoc"] = result["ml"]["feature_assoc_train"]
    return result


def selected_representatives() -> list[dict[str, Any]]:
    selected = []
    for rep in CANDIDATES:
        if rep.get("include_by_default") or INCLUDE_ML06_EXCLUDED:
            selected.append(dict(rep))
    return sorted(selected, key=lambda rep: (rep["phase_order"], rep["candidate_order"], rep["model_run_id"]))


def load_feature_combination_data() -> tuple[dict[str, dict[str, Any]], pd.DataFrame]:
    exp_data = {}
    skipped_rows = []
    for rep in selected_representatives():
        data = load_optional_artifacts(rep)
        if "metrics" not in data["ml"] and "train_summary" not in data["ml"]:
            reason = "metrics/train_summary 산출물이 모두 없음"
            if not SKIP_INVALID_RUNS:
                raise FileNotFoundError(f"{data['label']}: {reason} ({data['output_dir']})")
            skipped_rows.append({**rep, "reason": reason, "output_dir": str(data["output_dir"])})
            continue
        exp_data[f"{rep['ml_folder']}::{data['prefix']}"] = data
    return exp_data, pd.DataFrame(skipped_rows)


def sorted_exp_items(exp_data: dict[str, dict[str, Any]]) -> list[tuple[str, dict[str, Any]]]:
    return sorted(exp_data.items(), key=lambda item: (item[1]["rep"]["phase_order"], item[1]["rep"]["candidate_order"], item[1]["rep"]["model_run_id"]))


In [5]:
# ============================================================
# Summary table, file manifest, threshold sweep
# ============================================================
def train_val_gap(train_metrics: dict[str, Any], val_metrics: dict[str, Any], metric_key: str) -> Optional[float]:
    train_value = train_metrics.get(metric_key)
    val_value = val_metrics.get(metric_key)
    if train_value is None or val_value is None:
        return None
    return float(train_value) - float(val_value)


def build_summary_table(exp_data: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for _, data in sorted_exp_items(exp_data):
        rep = data["rep"]
        ml = data["ml"]
        metrics_raw = ml.get("metrics", {})
        metrics = metrics_payload(metrics_raw)
        train_metrics = metrics_payload(ml.get("metrics_train"))
        train_summary = train_summary_payload(ml)
        threshold = threshold_payload(ml)
        counts = confusion_counts(ml) or {}
        optuna_summary = ml.get("optuna_summary") if isinstance(ml.get("optuna_summary"), dict) else {}
        feature_columns = ml.get("feature_columns") if isinstance(ml.get("feature_columns"), list) else []
        curves, curve_source = available_learning_curves(train_summary)
        runtime_sec = metrics_raw.get("runtime_sec", {}) if isinstance(metrics_raw, dict) else {}
        feature_count = train_summary.get("feature_count")
        if feature_count is None and isinstance(metrics_raw, dict):
            feature_count = metrics_raw.get("feature_count")
        if feature_count is None:
            feature_count = len(feature_columns) or None
        rows.append({
            "phase": rep["phase"], "candidate": rep["candidate"], "family": rep["family"], "model_run_id": rep["model_run_id"],
            "feature_count": feature_count,
            "threshold_rule": threshold.get("threshold_strategy") or threshold.get("selection_metric") or "max_f1",
            "Threshold": metric_value(metrics, train_summary, "Threshold"),
            "F1": metric_value(metrics, train_summary, "F1"),
            "AUPRC": metric_value(metrics, train_summary, "AUPRC"),
            "Recall": metric_value(metrics, train_summary, "Recall"),
            "Precision": metric_value(metrics, train_summary, "Precision"),
            "TP": counts.get("tp"), "FP": counts.get("fp"), "FN": counts.get("fn"), "TN": counts.get("tn"),
            "train-val gap": train_val_gap(train_metrics, metrics, "f1"),
            "AUPRC train-val gap": train_val_gap(train_metrics, metrics, "average_precision"),
            "best_iteration": train_summary.get("best_iteration"),
            "training_time_sec": train_summary.get("training_time_sec"),
            "validate_time_sec": runtime_sec.get("total_validate_xgb") if isinstance(runtime_sec, dict) else None,
            "trials": optuna_summary.get("trial_count"),
            "best_trial": optuna_summary.get("best_trial_number"),
            "curve_metrics": ", ".join(sorted(curves)),
            "curve_source": curve_source if curves else "",
            "description": rep.get("description", ""),
        })
    return pd.DataFrame(rows)


def build_file_manifest_table(exp_data: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows = []
    artifact_names = list(ARTIFACT_SUFFIXES) + ["optuna_summary"]
    for _, data in sorted_exp_items(exp_data):
        loaded = {row["artifact"]: row for row in data["files"]}
        missing = {row["artifact"]: row for row in data["missing"]}
        for artifact in artifact_names:
            row = loaded.get(artifact)
            missing_row = missing.get(artifact)
            rows.append({
                "label": data["label"], "artifact": artifact, "loaded": row is not None,
                "size_mb": None if row is None else round(row["size_mb"], 3),
                "path": row.get("path") if row else (missing_row.get("path") if missing_row else None),
            })
    return pd.DataFrame(rows)


def prediction_scores_path(data: dict[str, Any]) -> Optional[Path]:
    path = data["ml"].get("prediction_scores_path")
    if isinstance(path, Path) and path.exists():
        return path
    metrics_raw = data["ml"].get("metrics", {})
    if isinstance(metrics_raw, dict):
        meta = metrics_raw.get("prediction_scores") or {}
        meta_path = meta.get("path") if isinstance(meta, dict) else None
        if meta_path and Path(meta_path).exists():
            return Path(meta_path)
    fallback = artifact_path(data["output_dir"], data["prefix"], ARTIFACT_SUFFIXES["prediction_scores_val"])
    return fallback if fallback.exists() else None


def prediction_score_columns(metrics_raw: dict[str, Any]) -> tuple[Optional[str], Optional[str]]:
    meta = metrics_raw.get("prediction_scores") or {}
    if not isinstance(meta, dict):
        return None, None
    return meta.get("score_column"), meta.get("label_column")


def load_prediction_score_frame(path: Path, metrics_raw: dict[str, Any]) -> pd.DataFrame:
    score_col, label_col = prediction_score_columns(metrics_raw)
    if score_col and label_col:
        try:
            return pd.read_parquet(path, columns=[score_col, label_col]).rename(columns={score_col: "score", label_col: "label"})
        except Exception as exc:
            print(f"column-select parquet load failed for {path.name}: {exc}. Fallback to full read.")
    df = pd.read_parquet(path)
    score_candidates = ["score", "prediction_score", "probability", "proba", "y_score", "pred_proba"]
    label_candidates = ["label", "y_true", "target", "is_laundering", "Is Laundering"]
    found_score = next((col for col in score_candidates if col in df.columns), None)
    found_label = next((col for col in label_candidates if col in df.columns), None)
    if found_score is None or found_label is None:
        raise ValueError(f"Cannot identify score/label columns in {path}. columns={list(df.columns)}")
    return df[[found_score, found_label]].rename(columns={found_score: "score", found_label: "label"})


def metrics_at_threshold(scores: pd.Series, labels: pd.Series, threshold: float, average_precision: Optional[float]) -> dict[str, Any]:
    pred = scores >= threshold
    label_bool = labels.astype(int) == 1
    tp = int((pred & label_bool).sum())
    fp = int((pred & ~label_bool).sum())
    fn = int((~pred & label_bool).sum())
    tn = int((~pred & ~label_bool).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"threshold": float(threshold), "F1": f1, "Precision": precision, "Recall": recall, "AP": average_precision, "TP": tp, "FP": fp, "FN": fn, "TN": tn, "predicted_positive": tp + fp}


def best_threshold_for_recall_floor(df: pd.DataFrame, floor: float, average_precision: Optional[float]) -> Optional[dict[str, Any]]:
    work = df[["score", "label"]].dropna().copy()
    if work.empty:
        return None
    work["label"] = work["label"].astype(int)
    work = work.sort_values("score", ascending=False, kind="mergesort").reset_index(drop=True)
    positives = int((work["label"] == 1).sum())
    negatives = len(work) - positives
    if positives == 0:
        return None
    work["tp"] = (work["label"] == 1).cumsum()
    work["fp"] = (work["label"] == 0).cumsum()
    grouped = work.groupby("score", sort=False).agg(tp=("tp", "max"), fp=("fp", "max")).reset_index()
    grouped["fn"] = positives - grouped["tp"]
    grouped["tn"] = negatives - grouped["fp"]
    grouped["Precision"] = grouped["tp"] / (grouped["tp"] + grouped["fp"])
    grouped["Recall"] = grouped["tp"] / positives
    grouped["F1"] = 2 * grouped["Precision"] * grouped["Recall"] / (grouped["Precision"] + grouped["Recall"])
    candidates = grouped[grouped["Recall"] >= floor].copy()
    if candidates.empty:
        return None
    row = candidates.sort_values(["F1", "score"], ascending=[False, False]).iloc[0]
    return {
        "threshold": float(row["score"]), "F1": float(row["F1"]), "Precision": float(row["Precision"]),
        "Recall": float(row["Recall"]), "AP": average_precision,
        "TP": int(row["tp"]), "FP": int(row["fp"]), "FN": int(row["fn"]), "TN": int(row["tn"]),
        "predicted_positive": int(row["tp"] + row["fp"]),
    }


def threshold_sweep_for_run(data: dict[str, Any], recall_floors: list[float]) -> list[dict[str, Any]]:
    path = prediction_scores_path(data)
    if path is None:
        return [{"phase": data["rep"]["phase"], "candidate": data["rep"]["candidate"], "threshold_rule": "missing_prediction_scores", "error": "prediction_scores_val.parquet 없음"}]
    metrics_raw = data["ml"].get("metrics", {})
    metrics = metrics_payload(metrics_raw)
    average_precision = metric_value(metrics, train_summary_payload(data["ml"]), "AUPRC")
    df = load_prediction_score_frame(path, metrics_raw if isinstance(metrics_raw, dict) else {})
    rows = []
    current_threshold = metric_value(metrics, train_summary_payload(data["ml"]), "Threshold")
    if current_threshold is not None:
        rows.append({"phase": data["rep"]["phase"], "candidate": data["rep"]["candidate"], "family": data["rep"]["family"], "threshold_rule": "max_f1", **metrics_at_threshold(df["score"], df["label"], float(current_threshold), average_precision)})
    for floor in recall_floors:
        row = best_threshold_for_recall_floor(df, floor, average_precision)
        rule = f"recall_floor_{floor:.2f}"
        if row is None:
            rows.append({"phase": data["rep"]["phase"], "candidate": data["rep"]["candidate"], "family": data["rep"]["family"], "threshold_rule": rule, "error": "threshold 없음"})
        else:
            rows.append({"phase": data["rep"]["phase"], "candidate": data["rep"]["candidate"], "family": data["rep"]["family"], "threshold_rule": rule, **row})
    return rows


def build_threshold_sweep_table(exp_data: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for _, data in sorted_exp_items(exp_data):
        rows.extend(threshold_sweep_for_run(data, RECALL_FLOORS))
    return pd.DataFrame(rows)


In [6]:
# ============================================================
# Figure builders
# ============================================================
METRIC_COLORS = {"F1": "#4f9cf9", "AUPRC": "#a78bfa", "Recall": "#34d399", "Precision": "#fbbf24"}
PHASE_COLORS = {"ML-06 fixed-param": "#4f9cf9", "ML-06 excluded": "#94a3b8", "ML-07 tuned": "#34d399"}


def make_metric_grouped_bar(summary_df: pd.DataFrame) -> Any:
    metric_order = ["F1", "AUPRC", "Recall", "Precision"]
    df = summary_df.melt(id_vars=["phase", "candidate", "family", "model_run_id", "description"], value_vars=metric_order, var_name="metric", value_name="value").dropna(subset=["value"])
    df["label"] = df["phase"] + " / " + df["candidate"]
    labels = (summary_df["phase"].astype(str) + " / " + summary_df["candidate"].astype(str)).tolist()
    if HAS_PLOTLY:
        fig = px.bar(df, x="label", y="value", color="metric", barmode="group", category_orders={"metric": metric_order, "label": labels}, color_discrete_map=METRIC_COLORS, hover_data={"phase": True, "candidate": True, "model_run_id": True, "description": True, "value": ":.6f"}, title="Metric Comparison")
        fig.update_layout(height=max(430, 26 * len(summary_df)), margin=dict(t=50, b=130), yaxis=dict(range=[0, 1]))
        fig.update_xaxes(tickangle=-35)
        return fig
    pivot = df.pivot_table(index="label", columns="metric", values="value")
    ax = pivot[metric_order].plot(kind="bar", figsize=(12, 5), color=[METRIC_COLORS[m] for m in metric_order])
    ax.set_ylim(0, 1)
    ax.grid(True, axis="y", alpha=0.25)
    fig = ax.get_figure()
    fig.tight_layout()
    return fig


def make_fixed_vs_tuned_fig(summary_df: pd.DataFrame) -> Any:
    comparable = summary_df[summary_df["family"].isin(["top10", "top12", "top12-plus-fanin", "top12-plus-acc"])].copy()
    comparable = comparable[comparable["phase"].isin(["ML-06 fixed-param", "ML-07 tuned"])]
    if comparable.empty:
        raise ValueError("fixed vs tuned 비교 대상이 없습니다.")
    df = comparable.melt(id_vars=["phase", "family", "candidate", "feature_count", "FP", "TP"], value_vars=["F1", "AUPRC", "Recall", "Precision"], var_name="metric", value_name="value").dropna(subset=["value"])
    if HAS_PLOTLY:
        fig = px.bar(df, x="family", y="value", color="phase", facet_col="metric", barmode="group", color_discrete_map=PHASE_COLORS, hover_data={"candidate": True, "feature_count": True, "TP": True, "FP": True, "value": ":.6f"}, title="ML-06 Fixed-param vs ML-07 Tuned")
        fig.update_yaxes(range=[0, 1])
        fig.update_layout(height=420, margin=dict(t=60, b=70))
        return fig
    pivot = df.pivot_table(index=["family", "metric"], columns="phase", values="value")
    ax = pivot.plot(kind="bar", figsize=(12, 5))
    ax.set_ylim(0, 1)
    ax.grid(True, axis="y", alpha=0.25)
    fig = ax.get_figure()
    fig.tight_layout()
    return fig


def make_feature_count_metric_scatter(summary_df: pd.DataFrame) -> Any:
    df = summary_df.melt(id_vars=["phase", "candidate", "family", "feature_count", "TP", "FP", "model_run_id"], value_vars=["F1", "AUPRC", "Recall", "Precision"], var_name="metric", value_name="value").dropna(subset=["feature_count", "value"])
    if HAS_PLOTLY:
        fig = px.scatter(df, x="feature_count", y="value", color="metric", symbol="phase", hover_name="candidate", color_discrete_map=METRIC_COLORS, hover_data={"family": True, "TP": True, "FP": True, "model_run_id": True, "value": ":.6f"}, title="Feature Count vs Metric")
        fig.update_layout(height=430, margin=dict(t=50, b=40), yaxis=dict(range=[0, 1]))
        return fig
    fig, ax = plt.subplots(figsize=(10, 5))
    for metric, sub in df.groupby("metric"):
        ax.scatter(sub["feature_count"], sub["value"], label=metric, alpha=0.8)
    ax.set_xlabel("feature_count")
    ax.set_ylabel("metric")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    return fig


def make_tp_fp_tradeoff_fig(summary_df: pd.DataFrame) -> Any:
    df = summary_df.dropna(subset=["TP", "FP"]).copy()
    if HAS_PLOTLY:
        fig = px.scatter(df, x="FP", y="TP", color="phase", size="F1", hover_name="candidate", text="candidate", color_discrete_map=PHASE_COLORS, hover_data={"F1": ":.6f", "AUPRC": ":.6f", "Recall": ":.6f", "Precision": ":.6f", "feature_count": True, "model_run_id": True}, title="TP / FP Trade-off")
        fig.update_traces(textposition="top center")
        fig.update_layout(height=470, margin=dict(t=50, b=40))
        return fig
    fig, ax = plt.subplots(figsize=(9, 5))
    for phase, sub in df.groupby("phase"):
        ax.scatter(sub["FP"], sub["TP"], label=phase, alpha=0.8)
        for _, row in sub.iterrows():
            ax.annotate(row["candidate"], (row["FP"], row["TP"]), fontsize=8)
    ax.set_xlabel("FP")
    ax.set_ylabel("TP")
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    return fig


def make_threshold_sweep_fig(threshold_df: pd.DataFrame) -> Any:
    df = threshold_df.dropna(subset=["FP", "TP", "F1"]).copy()
    if df.empty:
        raise ValueError("threshold sweep chart를 그릴 row가 없습니다.")
    if HAS_PLOTLY:
        fig = px.scatter(df, x="FP", y="TP", color="threshold_rule", symbol="phase", hover_name="candidate", size="F1", hover_data={"threshold": ":.6f", "Precision": ":.6f", "Recall": ":.6f", "AP": ":.6f"}, title="Threshold Sweep TP / FP")
        fig.update_layout(height=470, margin=dict(t=50, b=40))
        return fig
    fig, ax = plt.subplots(figsize=(9, 5))
    for rule, sub in df.groupby("threshold_rule"):
        ax.scatter(sub["FP"], sub["TP"], label=rule, alpha=0.8)
    ax.set_xlabel("FP")
    ax.set_ylabel("TP")
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    return fig


def make_dashboard_learning_curve_fig(train_vals: tuple[float, ...], val_vals: tuple[float, ...], metric_label: str, best_iter: int) -> Any:
    rows = ([{"Iteration": i + 1, "split": "train", metric_label: v} for i, v in enumerate(train_vals)] + [{"Iteration": i + 1, "split": "val", metric_label: v} for i, v in enumerate(val_vals)])
    df = pd.DataFrame(rows)
    if HAS_PLOTLY:
        fig = px.line(df, x="Iteration", y=metric_label, color="split", color_discrete_map={"train": "#4f9cf9", "val": "#ff7f0e"})
        fig.add_vline(x=best_iter + 1, line_dash="dash", line_color="#d62728", annotation_text="best", annotation_font_size=11)
        fig.update_layout(height=310, margin=dict(t=40, b=20), legend_title_text="")
        return fig
    fig, ax = plt.subplots(figsize=(10, 4))
    for split, color in [("train", "#4f9cf9"), ("val", "#ff7f0e")]:
        subset = df[df["split"] == split]
        ax.plot(subset["Iteration"], subset[metric_label], label=split, color=color, linewidth=2)
    ax.axvline(best_iter + 1, linestyle="--", color="#d62728", linewidth=1.2, label="best")
    ax.set_xlabel("Iteration")
    ax.set_ylabel(metric_label)
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    return fig


def make_cm_fig(tp: int, fn: int, fp: int, tn: int) -> Any:
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (tp + fn) if (tp + fn) > 0 else 0
    tnr = tn / (fp + tn) if (fp + tn) > 0 else 0
    z_text = [[f"{tp:,}<br>TPR {tpr:.1%}", f"{fn:,}<br>FNR {fnr:.1%}"], [f"{fp:,}<br>FPR {fpr:.1%}", f"{tn:,}<br>TNR {tnr:.1%}"]]
    if HAS_PLOTLY:
        fig = px.imshow([[tp, fn], [fp, tn]], x=["Pred Fraud", "Pred Normal"], y=["Actual Fraud", "Actual Normal"], color_continuous_scale="Blues", text_auto=False, title="")
        fig.update_traces(text=z_text, texttemplate="%{text}", textfont={"size": 11})
        fig.update_coloraxes(showscale=False)
        fig.update_layout(height=290, margin=dict(t=10, b=10))
        return fig
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow([[tp, fn], [fp, tn]], cmap="Blues")
    ax.set_xticks([0, 1], ["Pred Fraud", "Pred Normal"])
    ax.set_yticks([0, 1], ["Actual Fraud", "Actual Normal"])
    for r, row in enumerate(z_text):
        for c, text in enumerate(row):
            ax.text(c, r, text.replace("<br>", "\n"), ha="center", va="center", fontsize=10)
    fig.tight_layout()
    return fig


def make_fi_bar_fig(feature_importance: pd.DataFrame, top_n: int, high_first: bool) -> Any:
    if "importance_gain" not in feature_importance.columns:
        raise ValueError("feature_importance.csv에 importance_gain 컬럼이 없습니다.")
    fi_df = feature_importance.sort_values("importance_gain", ascending=not high_first).head(min(top_n, len(feature_importance))).sort_values("importance_gain").reset_index(drop=True)
    if HAS_PLOTLY:
        fig = px.bar(fi_df, x="importance_gain", y="feature", orientation="h", color="importance_gain", color_continuous_scale="Blues", labels={"importance_gain": "Gain", "feature": "Feature"})
        fig.update_traces(hovertemplate="<b>%{y}</b><br>Gain: %{x:,.1f}<extra></extra>")
        fig.update_coloraxes(showscale=False)
        fig.update_layout(height=max(400, len(fi_df) * 28), margin=dict(t=40, b=20))
        return fig
    fig, ax = plt.subplots(figsize=(10, max(4, len(fi_df) * 0.3)))
    ax.barh(fi_df["feature"], fi_df["importance_gain"], color="#4f9cf9")
    ax.set_xlabel("Gain")
    ax.grid(True, axis="x", alpha=0.25)
    fig.tight_layout()
    return fig


def make_assoc_heatmap(feature_assoc: dict[str, Any], top_features: tuple[str, ...]) -> Any:
    assoc = feature_assoc.get("association", {}) if isinstance(feature_assoc, dict) else {}
    feat_list = feature_assoc.get("features", []) if isinstance(feature_assoc, dict) else []
    matrix = assoc.get("matrix", [])
    metric_matrix = assoc.get("metric_matrix", [])
    if not feat_list or not matrix:
        raise ValueError("Association 데이터 없음")
    all_names = [f.get("name") for f in feat_list]
    if top_features:
        top_set = set(top_features)
        idx = [i for i, name in enumerate(all_names) if name in top_set]
        if not idx:
            raise ValueError("top feature와 association matrix의 feature 이름이 겹치지 않습니다")
        names = [all_names[i] for i in idx]
        matrix = [[matrix[r][c] for c in idx] for r in idx]
        metric_matrix = [[metric_matrix[r][c] for c in idx] for r in idx] if metric_matrix else []
    else:
        names = all_names
    if HAS_PLOTLY:
        custom = [[[metric_matrix[r][c] if metric_matrix else ""] for c in range(len(names))] for r in range(len(names))]
        fig = go.Figure(go.Heatmap(z=matrix, x=names, y=names, colorscale="RdBu_r", zmin=-1, zmax=1, customdata=custom, hovertemplate="<b>X: %{x}</b><br><b>Y: %{y}</b><br><b>%{customdata[0]}: %{z:.4f}</b><extra></extra>", showscale=False))
        fig.update_layout(height=max(400, len(names) * 28), margin=dict(t=40, b=20), yaxis=dict(scaleanchor="x", tickfont_size=9), xaxis=dict(showticklabels=False, ticks=""))
        return fig
    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(matrix, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_yticks(range(len(names)), names, fontsize=8)
    ax.set_xticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    return fig


In [7]:
# ============================================================
# Display helpers
# ============================================================
def display_metric_table(data: dict[str, Any]) -> None:
    summary_df = build_summary_table({"only": data})
    columns = ["phase", "candidate", "feature_count", "threshold_rule", "Threshold", "F1", "AUPRC", "Recall", "Precision", "TP", "FP", "FN", "best_iteration", "train-val gap", "AUPRC train-val gap", "training_time_sec", "validate_time_sec"]
    display(summary_df[[col for col in columns if col in summary_df.columns]])


def display_hyperparameters(train_summary: dict[str, Any]) -> None:
    params = train_summary.get("xgboost_params") or {}
    if not params:
        display(Markdown("Hyper Parameters: 파일 없음 또는 값 없음"))
        return
    display(pd.DataFrame([{"parameter": key, "value": value} for key, value in params.items() if value is not None]))


def display_learning_curves(data: dict[str, Any]) -> None:
    train_summary = train_summary_payload(data["ml"])
    curves, source = available_learning_curves(train_summary)
    if not curves:
        display(Markdown("학습 곡선 데이터 없음: `train_summary.xgboost_diagnostics.evals_result` 확인 필요"))
        return
    display(Markdown(f"curve source: `{source}`"))
    for metric_label in LEARNING_CURVE_METRICS:
        metric_key = dashboard_metric_key(metric_label)
        if metric_key not in curves:
            display(Markdown(f"- `{metric_label}` curve 없음. available={sorted(curves)}"))
            continue
        train_vals, val_vals = curves[metric_key]
        best_iter = int(train_summary.get("best_iteration") or 0)
        fig = make_dashboard_learning_curve_fig(train_vals, val_vals, metric_label, best_iter)
        if HAS_PLOTLY:
            fig.update_layout(title=f"{data['label']} {metric_label} Learning Curve")
        else:
            fig.axes[0].set_title(f"{data['label']} {metric_label} Learning Curve")
        display(fig)


def display_feature_importance(data: dict[str, Any]) -> tuple[str, ...]:
    feat_imp = data["ml"].get("feature_importance")
    if not isinstance(feat_imp, pd.DataFrame) or feat_imp.empty:
        display(Markdown("Feature importance 파일 없음"))
        return ()
    high_first = FEATURE_IMPORTANCE_SORT.lower() != "low"
    top_names = tuple(feat_imp.sort_values("importance_gain", ascending=not high_first).head(min(TOP_N_FEATURES, len(feat_imp)))["feature"].tolist())
    display(make_fi_bar_fig(feat_imp, TOP_N_FEATURES, high_first))
    return top_names


def display_feature_correlation(data: dict[str, Any], top_feature_names: tuple[str, ...]) -> None:
    if not SHOW_FEATURE_CORRELATION:
        display(Markdown("Feature Correlation: 설정에서 숨김"))
        return
    feature_assoc = data["ml"].get("feature_assoc")
    if not isinstance(feature_assoc, dict):
        display(Markdown("feature_assoc_mixed 파일 없음"))
        return
    split_label = feature_assoc.get("split", "?")
    methods = feature_assoc.get("association_methods", {})
    display(Markdown(f"Split: `{split_label}` | numeric-numeric: `{methods.get('numeric_numeric', '?')}` | cat-cat: `{methods.get('categorical_categorical', '?')}` | mixed: `{methods.get('numeric_categorical', '?')}`"))
    try:
        display(make_assoc_heatmap(feature_assoc, top_feature_names))
    except ValueError as exc:
        display(Markdown(f"Association 데이터 없음: `{exc}`"))


def display_score_profile(data: dict[str, Any]) -> None:
    if not SHOW_SCORE_PROFILE:
        display(Markdown("Score Profile: 설정에서 숨김"))
        return
    metrics_raw = data["ml"].get("metrics", {})
    if not isinstance(metrics_raw, dict):
        display(Markdown("Score profile 없음"))
        return
    score_profile = metrics_raw.get("score_profile") or {}
    prediction_scores = metrics_raw.get("prediction_scores") or {}
    rows = []
    for label, values in [("all", score_profile.get("probability_quantiles") or {}), ("positive", score_profile.get("positive_score_quantiles") or {}), ("negative", score_profile.get("negative_score_quantiles") or {})]:
        if values:
            row = {"group": label}
            row.update(values)
            rows.append(row)
    display(pd.DataFrame(rows) if rows else Markdown("Score quantile profile 없음"))
    meta_rows = []
    for key in ["predicted_positive_count", "predicted_positive_rate"]:
        if key in score_profile:
            meta_rows.append({"item": key, "value": score_profile[key]})
    for key in ["file_name", "rows", "score_column", "label_column", "split", "sampled"]:
        if key in prediction_scores:
            meta_rows.append({"item": f"prediction_scores.{key}", "value": prediction_scores[key]})
    if meta_rows:
        display(pd.DataFrame(meta_rows))
    path = prediction_scores_path(data)
    if path:
        display(Markdown(f"prediction score parquet는 threshold sweep 외에는 기본 미로드: `{path}`"))


def show_run_detail_section(data: dict[str, Any]) -> None:
    rep = data["rep"]
    display(Markdown(f"## {data['label']}"))
    display(Markdown(f"**Description**: {rep.get('description', '')}"))
    display(pd.DataFrame([{"prefix": data["prefix"], "output_dir": str(data["output_dir"])}]))
    display(Markdown("### Metrics"))
    display_metric_table(data)
    if SHOW_HYPERPARAMETERS:
        display(Markdown("### Hyper Parameters"))
        display_hyperparameters(train_summary_payload(data["ml"]))
    display(Markdown("### Learning Curve"))
    display_learning_curves(data)
    display(Markdown("### Confusion Matrix"))
    counts = confusion_counts(data["ml"])
    if counts is None:
        display(Markdown("Confusion matrix 파일 없음"))
    else:
        display(make_cm_fig(tp=counts["tp"], fn=counts["fn"], fp=counts["fp"], tn=counts["tn"]))
        display(pd.DataFrame([counts]))
    display(Markdown("### Feature Importance"))
    top_feature_names = display_feature_importance(data)
    display(Markdown("### Feature Correlation"))
    display_feature_correlation(data, top_feature_names)
    display(Markdown("### Score Profile"))
    display_score_profile(data)


def show_feature_combination_dashboard(exp_data: dict[str, dict[str, Any]], skipped_df: pd.DataFrame) -> None:
    display(Markdown("# ML-06 / ML-07 Feature Combination Dashboard"))
    display(Markdown(f"loaded runs = `{len(exp_data)}` | INCLUDE_ML06_EXCLUDED = `{INCLUDE_ML06_EXCLUDED}` | ENABLE_THRESHOLD_SWEEP = `{ENABLE_THRESHOLD_SWEEP}`"))
    display(Markdown("validation-only 비교입니다. test artifact는 읽지 않으며 threshold 선택에도 사용하지 않습니다."))
    summary_df = build_summary_table(exp_data)
    display(Markdown("## Summary"))
    display(summary_df if not summary_df.empty else Markdown("로드된 실험이 없습니다."))
    if not skipped_df.empty:
        display(Markdown("## Skipped Runs"))
        display(skipped_df)
    if SHOW_FILE_MANIFEST and exp_data:
        display(Markdown("## Artifact Manifest"))
        display(build_file_manifest_table(exp_data))
    if not summary_df.empty:
        display(Markdown("## Metric Comparison"))
        display(make_metric_grouped_bar(summary_df))
        display(Markdown("## ML-06 Fixed-param vs ML-07 Tuned"))
        try:
            display(make_fixed_vs_tuned_fig(summary_df))
        except ValueError as exc:
            display(Markdown(f"fixed vs tuned 표시 불가: `{exc}`"))
        display(Markdown("## Feature Count vs Metric"))
        display(make_feature_count_metric_scatter(summary_df))
        display(Markdown("## TP / FP Trade-off"))
        display(make_tp_fp_tradeoff_fig(summary_df))
    if ENABLE_THRESHOLD_SWEEP and exp_data:
        display(Markdown("## Threshold Sweep"))
        display(Markdown("`prediction_scores_val.parquet`를 후보별로 한 파일씩 순차 로드합니다."))
        threshold_df = build_threshold_sweep_table(exp_data)
        display(threshold_df)
        try:
            display(make_threshold_sweep_fig(threshold_df))
        except ValueError as exc:
            display(Markdown(f"threshold sweep chart 표시 불가: `{exc}`"))
    if SHOW_RUN_DETAILS:
        display(Markdown("# Candidate Details"))
        items = sorted_exp_items(exp_data)
        if RUN_DETAIL_LIMIT is not None:
            items = items[: int(RUN_DETAIL_LIMIT)]
        for _, data in items:
            show_run_detail_section(data)


In [8]:
# ============================================================
# 최종 실행 지점
# ============================================================
exp_data, skipped_df = load_feature_combination_data()
show_feature_combination_dashboard(exp_data, skipped_df)


# ML-06 / ML-07 Feature Combination Dashboard

loaded runs = `10` | INCLUDE_ML06_EXCLUDED = `False` | ENABLE_THRESHOLD_SWEEP = `True`

validation-only 비교입니다. test artifact는 읽지 않으며 threshold 선택에도 사용하지 않습니다.

## Summary

,phase,candidate,family,model_run_id,feature_count,threshold_rule,Threshold,F1,AUPRC,Recall,...,train-val gap,AUPRC train-val gap,best_iteration,training_time_sec,validate_time_sec,trials,best_trial,curve_metrics,curve_source,description
0,ML-06 fixed-param,full d02,full,d02-maxdepth4-fixparam-report,236,max_f1,0.965057,0.308612,0.243701,0.405355,...,-0.039674,0.065350,1463,106.211256,93.064304,NaN,NaN,"aucpr, logloss",xgboost_diagnostics.evals_result,ML-06 full fixed-param reference
1,ML-06 fixed-param,practical,practical,d02-maxdepth4-fixparam-practical,165,max_f1,0.959771,0.257990,0.175780,0.409972,...,-0.061778,0.024126,581,45.015308,70.435089,NaN,NaN,"aucpr, logloss",xgboost_diagnostics.evals_result,ML-06 practical reduced fixed-param baseline
2,ML-06 fixed-param,top10,top10,d02-maxdepth4-fixparam-practical-plus-rev27-ga...,175,max_f1,0.966271,0.277066,0.197514,0.359187,...,-0.041836,0.045459,899,57.987217,70.830403,NaN,NaN,"aucpr, logloss",xgboost_diagnostics.evals_result,ML-06 practical + rev27 gain top10 fixed-param
3,ML-06 fixed-param,top12,top12,d02-maxdepth4-fixparam-practical-plus-rev27-ga...,177,max_f1,0.959037,0.284902,0.213500,0.416436,...,-0.056999,0.056125,1100,66.674982,64.193028,NaN,NaN,"aucpr, logloss",xgboost_diagnostics.evals_result,ML-06 practical + rev27 gain top12 fixed-param
4,ML-06 fixed-param,fanin_only,top12-plus-fanin,d02-maxdepth4-fixparam-practical_top12_plus_fa...,181,max_f1,0.972104,0.302369,0.246274,0.371191,...,-0.015618,0.076263,1574,93.504915,71.325569,NaN,NaN,"aucpr, logloss",xgboost_diagnostics.evals_result,ML-06 top12 + remaining fan-in features fixed-...
5,ML-06 fixed-param,acc_only,top12-plus-acc,d02-maxdepth4-fixparam-practical_top12_plus_ac...,182,max_f1,0.949343,0.270993,0.201918,0.439520,...,-0.069533,0.031981,923,63.932266,68.882164,NaN,NaN,"aucpr, logloss",xgboost_diagnostics.evals_result,ML-06 top12 + remaining accountstats features ...
6,ML-07 tuned,top10,top10,ml07-d00-md4-top10-optuna-f1-t25,175,max_f1,0.960038,0.316334,0.252228,0.395199,...,-0.020953,0.105629,1399,82.535283,79.397956,25.0,14.0,"aucpr, logloss",xgboost_diagnostics.evals_result,ML-07 top10 Optuna validation-only tuned
7,ML-07 tuned,top12,top12,ml07-d00-md4-top12-optuna-f1-t25,177,max_f1,0.972719,0.321910,0.262351,0.385965,...,-0.007240,0.099258,1593,97.721497,98.690805,25.0,22.0,"aucpr, logloss",xgboost_diagnostics.evals_result,ML-07 top12 Optuna validation-only tuned
8,ML-07 tuned,top12-plus-fanin,top12-plus-fanin,ml07-d00-md4-top12-plus-fanin-optuna-f1-t25,181,max_f1,0.968784,0.330803,0.275435,0.382271,...,0.001920,0.112636,1791,103.458369,96.556669,25.0,5.0,"aucpr, logloss",xgboost_diagnostics.evals_result,ML-07 top12 plus fan-in Optuna validation-only...
9,ML-07 tuned,top12-plus-acc,top12-plus-acc,ml07-d00-md4-top12-plus-acc-optuna-f1-t25,182,max_f1,0.943649,0.314854,0.261364,0.409049,...,-0.008568,0.115084,1668,99.640265,82.259118,25.0,5.0,"aucpr, logloss",xgboost_diagnostics.evals_result,ML-07 top12 plus accountstats Optuna validatio...


...

Split: `val` | numeric-numeric: `pearson` | cat-cat: `cramers_v` | mixed: `correlation_ratio_eta`

### Score Profile

,group,q00,q01,q05,q10,q25,q50,q75,q90,q95,q99,q100
0,all,6.518228e-09,4.035565e-07,0.000001,0.000003,0.000010,0.000048,0.000334,0.006633,0.025962,0.318131,0.999946
1,positive,4.468594e-04,2.058273e-03,0.007357,0.021364,0.344187,0.896848,0.980553,0.996329,0.998418,0.999737,0.999946
2,negative,6.518228e-09,4.031963e-07,0.000001,0.000003,0.000010,0.000048,0.000331,0.006462,0.025266,0.281779,0.999458


,item,value
0,predicted_positive_count,1731
1,predicted_positive_rate,0.001704
2,prediction_scores.file_name,ml_07__r00__ml07-d00-md4-top12-plus-acc-optuna...
3,prediction_scores.rows,1015633
4,prediction_scores.score_column,score
5,prediction_scores.label_column,label
6,prediction_scores.split,val
7,prediction_scores.sampled,False


prediction score parquet는 threshold sweep 외에는 기본 미로드: `/mnt/d/AML_git/Graph_AML/ml/ml-07/ml_outputs/r00/ml_07__r00__ml07-d00-md4-top12-plus-acc-optuna-f1-t25_prediction_scores_val.parquet`